In [4]:
pip install eth_account web3 py-solc-x --break-system-packages

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: /usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from eth_account import Account
import secrets

# Generate a random 32-byte private key
# private_key = secrets.token_hex(32)
# print(private_key)

# 
private_key = "2f1a702f4c9b99ceb3ea1b801da326f98dd3c60b201ca01ebcb4fe76d15b5eaf"
print("Private Key:", private_key)

# Create account from the private key
account = Account.from_key(private_key)

print("Address:", account.address)
print("Public Key:", account.key.hex())


Private Key: 2f1a702f4c9b99ceb3ea1b801da326f98dd3c60b201ca01ebcb4fe76d15b5eaf
Address: 0x0037cfC76d08C39f3779a183c548F2FF3A2b9165
Public Key: 2f1a702f4c9b99ceb3ea1b801da326f98dd3c60b201ca01ebcb4fe76d15b5eaf


In [2]:
from web3 import Web3

# Replace with your Infura/Alchemy Sepolia endpoint
w3 = Web3(Web3.HTTPProvider("https://eth-sepolia.g.alchemy.com/v2/GBHT7Gy3kAPmnQF3OjZzW"))

# Check if connected
print("Connected:", w3.is_connected())

# Replace with your Sepolia testnet address
address = "0x0037cfC76d08C39f3779a183c548F2FF3A2b9165"

# Get balance in wei
balance_wei = w3.eth.get_balance(address)
print(balance_wei)
balance = w3.eth.get_balance(address)
print("Balance:", w3.from_wei(balance, 'ether'), "ETH")

Connected: True
49608000461103513
Balance: 0.049608000461103513 ETH


In [13]:
from solcx import compile_source, install_solc
import solcx
# Convert to ether
balance_eth = Web3.from_wei(balance_wei, 'ether')

print(f"Balance of {address}: {balance_eth} ETH")

# print(solcx.get_installable_solc_versions())
solcx.install_solc('0.8.5')

Balance of 0x0037cfC76d08C39f3779a183c548F2FF3A2b9165: 0.049899030291484912 ETH


<Version('0.8.5')>

In [19]:
import solcx                                                                                
solcx.install_solc('0.8.0') 
solcx.get_installable_solc_versions()

[<Version('0.8.34')>,
 <Version('0.8.33')>,
 <Version('0.8.32')>,
 <Version('0.8.31')>,
 <Version('0.8.30')>,
 <Version('0.8.29')>,
 <Version('0.8.28')>,
 <Version('0.8.27')>,
 <Version('0.8.26')>,
 <Version('0.8.25')>,
 <Version('0.8.24')>,
 <Version('0.8.23')>,
 <Version('0.8.22')>,
 <Version('0.8.21')>,
 <Version('0.8.20')>,
 <Version('0.8.19')>,
 <Version('0.8.18')>,
 <Version('0.8.17')>,
 <Version('0.8.16')>,
 <Version('0.8.15')>,
 <Version('0.8.14')>,
 <Version('0.8.13')>,
 <Version('0.8.12')>,
 <Version('0.8.11')>,
 <Version('0.8.10')>,
 <Version('0.8.9')>,
 <Version('0.8.8')>,
 <Version('0.8.7')>,
 <Version('0.8.6')>,
 <Version('0.8.5')>,
 <Version('0.8.4')>,
 <Version('0.8.3')>,
 <Version('0.8.2')>,
 <Version('0.8.1')>,
 <Version('0.8.0')>,
 <Version('0.7.6')>,
 <Version('0.7.5')>,
 <Version('0.7.4')>,
 <Version('0.7.3')>,
 <Version('0.7.2')>,
 <Version('0.7.1')>,
 <Version('0.7.0')>,
 <Version('0.6.12')>,
 <Version('0.6.11')>,
 <Version('0.6.10')>,
 <Version('0.6.9')>,
 <Vers

In [21]:
from solcx import compile_source, install_solc

# Install and set Solidity version
install_solc('0.8.0')

source_code = "None"
# Load Solidity source code from file
with open('zecdsa.sol', 'r') as file:
    source_code = file.read()

# Solidity source
contract_source =  source_code

# Compile
compiled = compile_source(contract_source, output_values=["abi", "bin"], solc_version="0.8.0")
print("Compiled")
contract_id, contract_interface = compiled.popitem()
abi = contract_interface["abi"]
bytecode = contract_interface["bin"]

# Deploy
contract = w3.eth.contract(abi=abi, bytecode=bytecode)
# tx_hash = contract.constructor().transact({'from': address})
tx = contract.constructor().build_transaction({
    'from': account.address,
    'nonce': w3.eth.get_transaction_count(account.address),
    'gas': 3000000,
    'gasPrice': w3.eth.gas_price,
    'chainId': 11155111
})
print(tx)

signed_tx = account.sign_transaction(tx)
tx_hash = w3.eth.send_raw_transaction(signed_tx.raw_transaction)
receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
tx_receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

print("Contract deployed at:", tx_receipt.contractAddress)
# Contract deployed at: 0x6C5297430852688B9D76d4af088AC68b54fd78Dc


Compiled
{'value': 0, 'from': '0x0037cfC76d08C39f3779a183c548F2FF3A2b9165', 'nonce': 2, 'gas': 3000000, 'gasPrice': 1000017, 'chainId': 11155111, 'data': '0x608060405234801561001057600080fd5b50610f72806100206000396000f3fe60806040526004361061007f5760003560e01c8063a7bb58031161004e578063a7bb58031461014c578063c64372861461018b578063d0e30db0146101c8578063fa540801146101d257610086565b806331e3ca0b1461008b578063448f30a3146100bb5780638d54a10c146100d257806397aba7f91461010f57610086565b3661008657005b600080fd5b6100a560048036038101906100a091906106dd565b61020f565b6040516100b29190610ba5565b60405180910390f35b3480156100c757600080fd5b506100d0610458565b005b3480156100de57600080fd5b506100f960048036038101906100f491906107eb565b61045a565b6040516101069190610ba5565b60405180910390f35b34801561011b57600080fd5b5061013660048036038101906101319190610814565b61047a565b6040516101439190610b8a565b60405180910390f35b34801561015857600080fd5b50610173600480360381019061016e9190610868565b6104e9565b60405161018293929190610bdb565b60405

In [25]:
# Send funds to contract
tx = {
    'from': account.address,
    'to': tx_receipt.contractAddress,
    'value': w3.to_wei(0.0001, 'ether'),
    'gas': 3000000,
    'gasPrice': w3.eth.gas_price,
    'nonce': w3.eth.get_transaction_count(account.address),
    'chainId': 11155111
}
# Sign and send
signed_tx = w3.eth.account.sign_transaction(tx, private_key)
tx_hash = w3.eth.send_raw_transaction(signed_tx.raw_transaction)
print(f"Transaction sent: {w3.to_hex(tx_hash)}")

Transaction sent: 0x48e738aa06c6db4d4eed51fe28c6dbf2a80a4572ff2bf1fe68460191d0d30d1b


In [27]:
from eth_account.messages import encode_defunct
from web3 import Web3

# print(tx_receipt.contractAddress)
# contractAddress = int("0x6C5297430852688B9D76d4af088AC68b54fd78Dc",16)
# contractAddress = "0x87e45c426EA700BD3CFFf790C655fC984d682Fe4"

# Message params
# recipient = Web3.to_checksum_address(account.address[:-1]+"7")
recipient = account.address
amount = 100
message = "Message"

# Hash the message
msg = Web3.solidity_keccak(['address', 'uint256', 'string'], [recipient, amount, message])
eth_msg = encode_defunct(primitive=msg)

# Sign with a private key
signed = w3.eth.account.sign_message(eth_msg, private_key=private_key)

contract_instance = w3.eth.contract(address=tx_receipt.contractAddress, abi=abi)
is_valid = contract_instance.functions.verify(
    account.address,
    recipient,
    amount,
    message,
    signed.signature
).call()
print("Signature is valid:", is_valid)

assert is_valid

# Call contract verify
contract_instance = w3.eth.contract(address=tx_receipt.contractAddress, abi=abi)
tx_est = contract_instance.functions.verify(
    account.address,
    recipient,
    amount,
    message,
    signed.signature
).build_transaction({
    'from': account.address,
    'nonce': w3.eth.get_transaction_count(account.address, 'pending'),
    'gas': 3000000,
    'gasPrice': w3.eth.gas_price,
    'chainId': 11155111
})
gasEstimate = w3.eth.estimate_gas(tx_est)
print("Total Gas Estimate (units):", gasEstimate)

signed_tx = w3.eth.account.sign_transaction(tx_est, private_key)
tx_hash = w3.eth.send_raw_transaction(signed_tx.raw_transaction)
# Assume you already have the receipt after sending a transaction
tx_receipt_transact = w3.eth.wait_for_transaction_receipt(tx_hash)
print(f"Transaction sent: {w3.to_hex(tx_hash)}")
print(tx_receipt_transact)

contract_instance = w3.eth.contract(address=tx_receipt.contractAddress, abi=abi)
tx_est = contract_instance.functions.nothing(
).build_transaction({
    'from': account.address,
    'nonce': w3.eth.get_transaction_count(account.address),
    'gas': 3000000,
    'gasPrice': w3.eth.gas_price,
    'chainId': 11155111
})
gasEstimate = w3.eth.estimate_gas(tx_est)
print("Base Case (emtpy):",gasEstimate)

assert False

# Assume you already have the receipt after sending a transaction
tx_receipt_transact = w3.eth.wait_for_transaction_receipt(tx_hash)

# Get gas used and effective gas price (EIP-1559 compatible)
gas_used = tx_receipt_transact['gasUsed']
effective_gas_price = tx_receipt_transact['effectiveGasPrice']  # In wei

# Total cost in wei
total_cost_wei = gas_used * effective_gas_price

# Convert to ether
total_cost_eth = Web3.fromWei(total_cost_wei, 'ether')

print(f"Gas Used: {gas_used}")
print(f"Gas Price: {Web3.fromWei(effective_gas_price, 'gwei')} Gwei")
print(f"Total Transaction Cost: {total_cost_eth} ETH")


Signature is valid: True
Total Gas Estimate (units): 64749
Transaction sent: 0xd880b0ac266a9a4fb43b4ec69b8bfd096d23dd79541bba63741d8c7b91bdd9ee
AttributeDict({'type': 0, 'status': 1, 'cumulativeGasUsed': 19556014, 'logs': [], 'logsBloom': HexBytes('0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000'), 'transactionHash': HexBytes('0xd880b0ac266a9a4fb43b4ec69b8bfd096d23dd79541bba63741d8c7b91bdd9ee'), 'transactionIndex': 125, 'blockHash': HexBytes('0xe308a55c9202d30d76b69dbcfa6ff22a15d3d4a27fb00e74d69a9f72117df2fa'), 'blockNumber': 1

AssertionError: 